<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
retrieval_failure_queries = [
    "What is the refund policy for cancelled orders?",
    "Which database is used to store user authentication information?",
    "What is the maximum file size allowed for uploads?"
]

In [ ]:
context_overflow_queries = [
    "Give me a complete summary of all the system features and their implementation details.",
    "Explain everything about the application's architecture, database, authentication, APIs, and deployment.",
    "Describe all the technical requirements and implementation decisions mentioned in the documents."
]

In [ ]:
answer_context_mismatch_queries = [
    "According to the retrieved documents, what programming language is used?",
    "According to the documentation, what authentication method is implemented?",
    "According to the provided context, what is the purpose of the API?"
]

In [ ]:
vague_context_queries = [
    "Tell me about the system.",
    "How does it work?",
    "What are its main features?"
]

In [ ]:
wrong_answer_generation_queries = [
    "What is the exact maximum number of users supported?",
    "What is the exact database name mentioned in the documentation?",
    "What is the exact API endpoint used for authentication?"
]

In [ ]:
test_queries = [
    # Retrieval failure
    {
        "query": "What is the refund policy for cancelled orders?",
        "target_failure": "retrieval_failure"
    },
    {
        "query": "Which database is used to store user authentication information?",
        "target_failure": "retrieval_failure"
    },
    {
        "query": "What is the maximum file size allowed for uploads?",
        "target_failure": "retrieval_failure"
    },

    # Context window overflow
    {
        "query": "Give me a complete summary of all the system features and their implementation details.",
        "target_failure": "context_window_overflow"
    },
    {
        "query": "Explain everything about the application's architecture, database, authentication, APIs, and deployment.",
        "target_failure": "context_window_overflow"
    },
    {
        "query": "Describe all the technical requirements and implementation decisions mentioned in the documents.",
        "target_failure": "context_window_overflow"
    },

    # Answer-context mismatch
    {
        "query": "According to the retrieved documents, what programming language is used?",
        "target_failure": "answer_context_mismatch"
    },
    {
        "query": "According to the documentation, what authentication method is implemented?",
        "target_failure": "answer_context_mismatch"
    },
    {
        "query": "According to the provided context, what is the purpose of the API?",
        "target_failure": "answer_context_mismatch"
    },

    # Vague context
    {
        "query": "Tell me about the system.",
        "target_failure": "vague_context"
    },
    {
        "query": "How does it work?",
        "target_failure": "vague_context"
    },
    {
        "query": "What are its main features?",
        "target_failure": "vague_context"
    },

    # Correct chunk but wrong answer
    {
        "query": "What is the exact maximum number of users supported?",
        "target_failure": "wrong_answer_generation"
    },
    {
        "query": "What is the exact database name mentioned in the documentation?",
        "target_failure": "wrong_answer_generation"
    },
    {
        "query": "What is the exact API endpoint used for authentication?",
        "target_failure": "wrong_answer_generation"
    }
]

print("Total test queries:", len(test_queries))

Total test queries: 15


In [ ]:
def diagnostic_rag(query, top_k=3):
    query_embedding = embedder.embed_query(query)
    distances, indices = vector_store.index.search(
        np.array([query_embedding]).astype("float32"),
        top_k
    )
    retrieved_chunks = []
    for i, idx in enumerate(indices[0]):
        if idx != -1:
            chunk = vector_store.documents[idx]
            retrieved_chunks.append({
                "rank": i + 1,
                "content": chunk,
                "similarity_score": float(distances[0][i])
            })
    context = "\n\n".join(
        item["content"] for item in retrieved_chunks
    )
    prompt = f"""
You are a helpful RAG assistant.
Answer the user's question using ONLY the provided context.
If the answer cannot be found in the context, say:
"I don't have enough information in the provided context."
Context:
{context}
Question:
{query}
Answer:
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content
    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer
    }

In [ ]:
print([name for name in globals() if not name.startswith("_")])

['In', 'Out', 'get_ipython', 'exit', 'quit', 'retrieval_failure_queries', 'context_overflow_queries', 'answer_context_mismatch_queries', 'vague_context_queries', 'wrong_answer_generation_queries', 'test_queries', 'diagnostic_rag']


In [12]:
!pip install -q sentence-transformers faiss-cpu transformers torch

In [13]:
import numpy as np
import faiss
import json
from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [14]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)
print("Models loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [15]:
documents = [
    "The application uses Python as its primary programming language for backend development.",
    "The application uses React for building the frontend user interface.",
    "PostgreSQL is used as the primary relational database for storing application data.",
    "Redis is used for caching frequently accessed data and improving application performance.",
    "User authentication is implemented using JSON Web Tokens (JWT).",
    "The authentication API endpoint is /api/auth/login.",
    "The registration API endpoint is /api/auth/register.",
    "Users can upload PDF files with a maximum file size of 10 MB.",
    "Uploaded PDF documents are processed and divided into smaller text chunks.",
    "The chunking system uses a chunk size of 500 characters with an overlap of 50 characters.",
    "FAISS is used as the vector database for storing and searching document embeddings.",
    "The semantic search system retrieves the top 3 most similar chunks for each query.",
    "The application supports a maximum of 1,000 concurrent users.",
    "The application provides document upload, semantic search, authentication, and course generation features.",
    "The refund policy allows users to request a refund within 7 days of purchase.",
    "Refund requests are reviewed by the support team before the refund is processed.",
    "The application is deployed using Docker containers.",
    "The backend exposes REST APIs for communication between the frontend and backend.",
    "The system uses cosine similarity to compare query and document embeddings.",
    "Documents are preprocessed before embedding by removing unnecessary whitespace and normalizing text."
]
print("Number of documents:", len(documents))

Number of documents: 20


In [16]:
document_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True
)
print("Embedding shape:", document_embeddings.shape)

Embedding shape: (20, 384)


In [17]:
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(
    document_embeddings.astype("float32")
)
print("FAISS index created!")
print("Number of vectors:", index.ntotal)

FAISS index created!
Number of vectors: 20


In [18]:
def diagnostic_rag(query, top_k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        retrieved_chunks.append({
            "rank": rank,
            "content": documents[idx],
            "similarity_score": float(distance)
        })
    context = "\n\n".join(
        chunk["content"]
        for chunk in retrieved_chunks
    )
    prompt = f"""
Answer the question using ONLY the context below.
If the answer is not present in the context, say:
"I don't have enough information in the provided context."
Context:
{context}
Question:
{query}
Answer:
"""
    response = generator(
        prompt,
        max_new_tokens=100
    )
    answer = response[0]["generated_text"]
    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer
    }

In [21]:
def generate_answer(context, query):
    """
    Simple local answer generator.
    Finds the most relevant sentence from the retrieved context.
    """
    sentences = []
    for chunk in context.split("\n\n"):
        sentences.extend(chunk.split("."))
    query_words = set(query.lower().split())
    best_sentence = ""
    best_score = 0
    for sentence in sentences:
        sentence_words = set(sentence.lower().split())
        score = len(query_words.intersection(sentence_words))
        if score > best_score:
            best_score = score
            best_sentence = sentence.strip()
    if best_sentence:
        return best_sentence + "."
    return "I don't have enough information in the provided context."
print("Local answer generator ready!")

Local answer generator ready!


In [23]:
def diagnostic_rag(query, top_k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        retrieved_chunks.append({
            "rank": rank,
            "content": documents[idx],
            "similarity_score": float(distance)
        })
    context = "\n\n".join(
        chunk["content"]
        for chunk in retrieved_chunks
    )
    answer = generate_answer(
        context,
        query
    )
    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer
    }

In [24]:
result = diagnostic_rag(
    "What database is used by the application?"
)
print("QUERY:")
print(result["query"])
print("\nRETRIEVED CHUNKS:")
for chunk in result["retrieved_chunks"]:
    print("\nRank:", chunk["rank"])
    print("Score:", chunk["similarity_score"])
    print("Content:", chunk["content"])
print("\nFINAL ANSWER:")
print(result["answer"])

QUERY:
What database is used by the application?

RETRIEVED CHUNKS:

Rank: 1
Score: 0.6241037845611572
Content: PostgreSQL is used as the primary relational database for storing application data.

Rank: 2
Score: 1.0454661846160889
Content: The application provides document upload, semantic search, authentication, and course generation features.

Rank: 3
Score: 1.1311616897583008
Content: Redis is used for caching frequently accessed data and improving application performance.

FINAL ANSWER:
PostgreSQL is used as the primary relational database for storing application data.


In [25]:
all_results = []
for i, test in enumerate(test_queries, start=1):
    print(f"Running query {i}/15...")
    try:
        result = diagnostic_rag(test["query"])
        result["test_id"] = i
        result["target_failure"] = test["target_failure"]
        all_results.append(result)
    except Exception as e:
        all_results.append({
            "test_id": i,
            "query": test["query"],
            "target_failure": test["target_failure"],
            "error": str(e)
        })
print("\nCompleted:", len(all_results), "queries")

Running query 1/15...
Running query 2/15...
Running query 3/15...
Running query 4/15...
Running query 5/15...
Running query 6/15...
Running query 7/15...
Running query 8/15...
Running query 9/15...
Running query 10/15...
Running query 11/15...
Running query 12/15...
Running query 13/15...
Running query 14/15...
Running query 15/15...

Completed: 15 queries


In [26]:
for result in all_results:
    if "error" in result:
        print("ERROR in query", result["test_id"])
        print(result["error"])

In [27]:
with open("rag_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)
print("Saved: rag_results.json")

Saved: rag_results.json


In [28]:
for result in all_results:
    print("=" * 80)
    print("TEST ID:", result["test_id"])
    print("QUERY:", result["query"])
    print("TARGET FAILURE:", result["target_failure"])
    print("\nRETRIEVED CHUNKS:")
    for chunk in result["retrieved_chunks"]:
        print(
            f"\nRank {chunk['rank']} "
            f"| L2 Distance: {chunk['similarity_score']:.4f}"
        )
        print(chunk["content"])
    print("\nGENERATED ANSWER:")
    print(result["answer"])
    print()

TEST ID: 1
QUERY: What is the refund policy for cancelled orders?
TARGET FAILURE: retrieval_failure

RETRIEVED CHUNKS:

Rank 1 | L2 Distance: 0.7644
The refund policy allows users to request a refund within 7 days of purchase.

Rank 2 | L2 Distance: 0.8274
Refund requests are reviewed by the support team before the refund is processed.

Rank 3 | L2 Distance: 1.8138
Redis is used for caching frequently accessed data and improving application performance.

GENERATED ANSWER:
The refund policy allows users to request a refund within 7 days of purchase.

TEST ID: 2
QUERY: Which database is used to store user authentication information?
TARGET FAILURE: retrieval_failure

RETRIEVED CHUNKS:

Rank 1 | L2 Distance: 0.9136
User authentication is implemented using JSON Web Tokens (JWT).

Rank 2 | L2 Distance: 1.0024
PostgreSQL is used as the primary relational database for storing application data.

Rank 3 | L2 Distance: 1.1740
The authentication API endpoint is /api/auth/login.

GENERATED ANSWER:

In [29]:
expected_answers = {
    1: "7 days",
    2: "PostgreSQL",
    3: "10 MB",
    4: "application features",
    5: "architecture database authentication APIs deployment",
    6: "technical requirements",
    7: "Python",
    8: "JWT",
    9: "REST APIs",
    10: "system",
    11: "work",
    12: "features",
    13: "1,000",
    14: "PostgreSQL",
    15: "/api/auth/login"
}

In [30]:
def retrieval_score(result, expected):
    """
    Score retrieval from 1 to 5.
    """
    retrieved_text = " ".join(
        chunk["content"].lower()
        for chunk in result["retrieved_chunks"]
    )
    expected_words = expected.lower().split()
    matches = sum(
        1 for word in expected_words
        if word in retrieved_text
    )
    if matches == len(expected_words):
        return 5
    elif matches >= max(1, len(expected_words) * 0.75):
        return 4
    elif matches >= max(1, len(expected_words) * 0.5):
        return 3
    elif matches > 0:
        return 2
    else:
        return 1
def answer_score(result, expected):
    """
    Score generated answer from 1 to 5.
    """
    answer = result["answer"].lower()
    expected = expected.lower()
    expected_words = expected.split()
    matches = sum(
        1 for word in expected_words
        if word in answer
    )
    if matches == len(expected_words):
        return 5
    elif matches >= max(1, len(expected_words) * 0.75):
        return 4
    elif matches >= max(1, len(expected_words) * 0.5):
        return 3
    elif matches > 0:
        return 2
    else:
        return 1

In [31]:
scorecard = []
for result in all_results:
    test_id = result["test_id"]
    expected = expected_answers[test_id]
    r_score = retrieval_score(
        result,
        expected
    )
    a_score = answer_score(
        result,
        expected
    )
    scorecard.append({
        "test_id": test_id,
        "query": result["query"],
        "target_failure": result["target_failure"],
        "retrieval_quality": r_score,
        "answer_quality": a_score,
        "answer": result["answer"]
    })
print("Scorecard created for", len(scorecard), "queries")

Scorecard created for 15 queries


In [32]:
import pandas as pd
scorecard_df = pd.DataFrame(scorecard)
scorecard_df

,test_id,query,target_failure,retrieval_quality,answer_quality,answer
0,1,What is the refund policy for cancelled orders?,retrieval_failure,5,5,The refund policy allows users to request a re...
1,2,Which database is used to store user authentic...,retrieval_failure,5,1,User authentication is implemented using JSON ...
2,3,What is the maximum file size allowed for uplo...,retrieval_failure,5,5,Users can upload PDF files with a maximum file...
3,4,Give me a complete summary of all the system f...,context_window_overflow,5,5,"The application provides document upload, sema..."
4,5,Explain everything about the application's arc...,context_window_overflow,2,2,"The application provides document upload, sema..."
5,6,Describe all the technical requirements and im...,context_window_overflow,1,1,"The application provides document upload, sema..."
6,7,"According to the retrieved documents, what pro...",answer_context_mismatch,5,5,The application uses Python as its primary pro...
7,8,"According to the documentation, what authentic...",answer_context_mismatch,5,1,The authentication API endpoint is /api/auth/l...
8,9,"According to the provided context, what is the...",answer_context_mismatch,5,1,The registration API endpoint is /api/auth/reg...
9,10,Tell me about the system.,vague_context,1,1,The application uses React for building the fr...


In [33]:
average_retrieval = scorecard_df["retrieval_quality"].mean()
average_answer = scorecard_df["answer_quality"].mean()
print(f"Average Retrieval Quality: {average_retrieval:.2f}/5")
print(f"Average Answer Quality: {average_answer:.2f}/5")

Average Retrieval Quality: 3.73/5
Average Answer Quality: 2.93/5


In [34]:
def classify_failure(result, expected):
    retrieved_text = " ".join(
        chunk["content"].lower()
        for chunk in result["retrieved_chunks"]
    )
    answer = result["answer"].lower()
    expected_words = expected.lower().split()
    retrieved_matches = sum(
        word in retrieved_text
        for word in expected_words
    )
    answer_matches = sum(
        word in answer
        for word in expected_words
    )
    if retrieved_matches == 0:
        return "retrieval_failure"
    if retrieved_matches > 0 and answer_matches == 0:
        return "answer_context_mismatch"
    if retrieved_matches > 0 and answer_matches > 0:
        return "correct_retrieval"
    return "unknown"

In [35]:
failure_analysis = []
for result in all_results:
    test_id = result["test_id"]
    expected = expected_answers[test_id]
    failure_type = classify_failure(
        result,
        expected
    )
    failure_analysis.append({
        "test_id": test_id,
        "query": result["query"],
        "target_failure": result["target_failure"],
        "actual_failure": failure_type
    })
failure_df = pd.DataFrame(failure_analysis)
failure_df

,test_id,query,target_failure,actual_failure
0,1,What is the refund policy for cancelled orders?,retrieval_failure,correct_retrieval
1,2,Which database is used to store user authentic...,retrieval_failure,answer_context_mismatch
2,3,What is the maximum file size allowed for uplo...,retrieval_failure,correct_retrieval
3,4,Give me a complete summary of all the system f...,context_window_overflow,correct_retrieval
4,5,Explain everything about the application's arc...,context_window_overflow,correct_retrieval
5,6,Describe all the technical requirements and im...,context_window_overflow,retrieval_failure
6,7,"According to the retrieved documents, what pro...",answer_context_mismatch,correct_retrieval
7,8,"According to the documentation, what authentic...",answer_context_mismatch,answer_context_mismatch
8,9,"According to the provided context, what is the...",answer_context_mismatch,answer_context_mismatch
9,10,Tell me about the system.,vague_context,retrieval_failure


In [36]:
def create_diagnosis(row):
    failure = row["actual_failure"]
    if failure == "retrieval_failure":
        return (
            "The relevant information was not present in the retrieved "
            "top-k chunks, so the retrieval stage failed."
        )
    elif failure == "answer_context_mismatch":
        return (
            "The relevant information was retrieved, but the generated "
            "answer did not correctly use that information."
        )
    elif failure == "correct_retrieval":
        return (
            "The relevant chunk was retrieved and the answer was supported "
            "by the retrieved context."
        )
    else:
        return (
            "The result requires manual inspection because the automatic "
            "classifier could not determine the failure type."
        )
failure_df["diagnosis"] = failure_df.apply(
    create_diagnosis,
    axis=1
)
failure_df

,test_id,query,target_failure,actual_failure,diagnosis
0,1,What is the refund policy for cancelled orders?,retrieval_failure,correct_retrieval,The relevant chunk was retrieved and the answe...
1,2,Which database is used to store user authentic...,retrieval_failure,answer_context_mismatch,"The relevant information was retrieved, but th..."
2,3,What is the maximum file size allowed for uplo...,retrieval_failure,correct_retrieval,The relevant chunk was retrieved and the answe...
3,4,Give me a complete summary of all the system f...,context_window_overflow,correct_retrieval,The relevant chunk was retrieved and the answe...
4,5,Explain everything about the application's arc...,context_window_overflow,correct_retrieval,The relevant chunk was retrieved and the answe...
5,6,Describe all the technical requirements and im...,context_window_overflow,retrieval_failure,The relevant information was not present in th...
6,7,"According to the retrieved documents, what pro...",answer_context_mismatch,correct_retrieval,The relevant chunk was retrieved and the answe...
7,8,"According to the documentation, what authentic...",answer_context_mismatch,answer_context_mismatch,"The relevant information was retrieved, but th..."
8,9,"According to the provided context, what is the...",answer_context_mismatch,answer_context_mismatch,"The relevant information was retrieved, but th..."
9,10,Tell me about the system.,vague_context,retrieval_failure,The relevant information was not present in th...


In [37]:
def get_retrieved_text(result):
    return " ".join(
        chunk["content"]
        for chunk in result["retrieved_chunks"]
    ).lower()
def contains_expected(result, expected):
    retrieved_text = get_retrieved_text(result)
    expected_words = expected.lower().split()
    matches = sum(
        word in retrieved_text
        for word in expected_words
    )
    return matches > 0

In [38]:
vague_keywords = [
    "tell me about",
    "how does it work",
    "main features"
]
for result in all_results:
    query = result["query"].lower()
    if any(keyword in query for keyword in vague_keywords):
        print("=" * 70)
        print("QUERY:", result["query"])
        print("\nRETRIEVED CONTEXT:")
        for chunk in result["retrieved_chunks"]:
            print("-", chunk["content"])
        print("\nANSWER:")
        print(result["answer"])

QUERY: Tell me about the system.

RETRIEVED CONTEXT:
- The application uses React for building the frontend user interface.
- The application supports a maximum of 1,000 concurrent users.
- The backend exposes REST APIs for communication between the frontend and backend.

ANSWER:
The application uses React for building the frontend user interface.
QUERY: How does it work?

RETRIEVED CONTEXT:
- User authentication is implemented using JSON Web Tokens (JWT).
- The application uses React for building the frontend user interface.
- The backend exposes REST APIs for communication between the frontend and backend.

ANSWER:
I don't have enough information in the provided context.
QUERY: What are its main features?

RETRIEVED CONTEXT:
- The application uses React for building the frontend user interface.
- The application uses Python as its primary programming language for backend development.
- PostgreSQL is used as the primary relational database for storing application data.

ANSWER:
The ap

In [39]:
for result in all_results:
    test_id = result["test_id"]
    if test_id in [7, 8, 9, 13, 14, 15]:
        expected = expected_answers[test_id]
        retrieved = contains_expected(
            result,
            expected
        )
        answer = result["answer"].lower()
        answer_correct = any(
            word in answer
            for word in expected.lower().split()
        )
        if retrieved and not answer_correct:
            print("=" * 70)
            print("TEST ID:", test_id)
            print("QUERY:", result["query"])
            print("\nEXPECTED:")
            print(expected)
            print("\nRETRIEVED:")
            for chunk in result["retrieved_chunks"]:
                print("-", chunk["content"])
            print("\nGENERATED ANSWER:")
            print(result["answer"])

TEST ID: 8
QUERY: According to the documentation, what authentication method is implemented?

EXPECTED:
JWT

RETRIEVED:
- User authentication is implemented using JSON Web Tokens (JWT).
- The authentication API endpoint is /api/auth/login.
- The registration API endpoint is /api/auth/register.

GENERATED ANSWER:
The authentication API endpoint is /api/auth/login.
TEST ID: 9
QUERY: According to the provided context, what is the purpose of the API?

EXPECTED:
REST APIs

RETRIEVED:
- The backend exposes REST APIs for communication between the frontend and backend.
- User authentication is implemented using JSON Web Tokens (JWT).
- The registration API endpoint is /api/auth/register.

GENERATED ANSWER:
The registration API endpoint is /api/auth/register.


In [40]:
def context_size(query, top_k=20):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(
        query_embedding,
        min(top_k, len(documents))
    )
    context = "\n\n".join(
        documents[idx]
        for idx in indices[0]
        if idx != -1
    )
    return len(context), context
for result in all_results:
    size, _ = context_size(
        result["query"],
        top_k=20
    )
    print(
        f"Test {result['test_id']}: "
        f"{size} characters of context"
    )

Test 1: 1553 characters of context
Test 2: 1553 characters of context
Test 3: 1553 characters of context
Test 4: 1553 characters of context
Test 5: 1553 characters of context
Test 6: 1553 characters of context
Test 7: 1553 characters of context
Test 8: 1553 characters of context
Test 9: 1553 characters of context
Test 10: 1553 characters of context
Test 11: 1553 characters of context
Test 12: 1553 characters of context
Test 13: 1553 characters of context
Test 14: 1553 characters of context
Test 15: 1553 characters of context


In [41]:
def retrieve_with_threshold(query, top_k=3, threshold=1.2):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        if idx == -1:
            continue
        if distance <= threshold:
            retrieved_chunks.append({
                "rank": rank,
                "content": documents[idx],
                "similarity_score": float(distance)
            })
    return retrieved_chunks

In [42]:
test_query = "What is the maximum file size allowed for uploads?"
results = retrieve_with_threshold(
    test_query,
    top_k=3,
    threshold=1.2
)
print("Query:", test_query)
for chunk in results:
    print("\nRank:", chunk["rank"])
    print("Distance:", chunk["similarity_score"])
    print("Content:", chunk["content"])

Query: What is the maximum file size allowed for uploads?

Rank: 1
Distance: 0.5105496644973755
Content: Users can upload PDF files with a maximum file size of 10 MB.

Rank: 2
Distance: 1.1270681619644165
Content: The chunking system uses a chunk size of 500 characters with an overlap of 50 characters.

Rank: 3
Distance: 1.1680350303649902
Content: Uploaded PDF documents are processed and divided into smaller text chunks.


In [43]:
def create_chunks(text, chunk_size=150, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

In [44]:
improved_chunks = []
for document in documents:
    improved_chunks.extend(
        create_chunks(
            document,
            chunk_size=150,
            overlap=50
        )
    )
print("Original documents:", len(documents))
print("Improved chunks:", len(improved_chunks))

Original documents: 20
Improved chunks: 21


In [45]:
improved_embeddings = embedder.encode(
    improved_chunks,
    convert_to_numpy=True
)
print("Embedding shape:", improved_embeddings.shape)

Embedding shape: (21, 384)


In [46]:
improved_index = faiss.IndexFlatL2(
    improved_embeddings.shape[1]
)
improved_index.add(
    improved_embeddings.astype("float32")
)
print("Improved FAISS index created!")
print("Number of vectors:", improved_index.ntotal)

Improved FAISS index created!
Number of vectors: 21


In [47]:
query = "What is the maximum file size allowed for PDF uploads?"

In [48]:
query_embedding = embedder.encode(
    [query],
    convert_to_numpy=True
).astype("float32")
old_distances, old_indices = index.search(
    query_embedding,
    3
)
print("OLD CHUNKING")
print("=" * 50)
for rank, (idx, distance) in enumerate(
    zip(old_indices[0], old_distances[0]),
    start=1
):
    print(f"\nRank {rank}")
    print("Distance:", distance)
    print("Content:", documents[idx])

OLD CHUNKING

Rank 1
Distance: 0.30513924
Content: Users can upload PDF files with a maximum file size of 10 MB.

Rank 2
Distance: 0.86261153
Content: Uploaded PDF documents are processed and divided into smaller text chunks.

Rank 3
Distance: 1.229543
Content: The chunking system uses a chunk size of 500 characters with an overlap of 50 characters.


In [49]:
new_distances, new_indices = improved_index.search(
    query_embedding,
    3
)
print("\n\nNEW CHUNKING")
print("=" * 50)
for rank, (idx, distance) in enumerate(
    zip(new_indices[0], new_distances[0]),
    start=1
):
    print(f"\nRank {rank}")
    print("Distance:", distance)
    print("Content:", improved_chunks[idx])



NEW CHUNKING

Rank 1
Distance: 0.30513924
Content: Users can upload PDF files with a maximum file size of 10 MB.

Rank 2
Distance: 0.86261153
Content: Uploaded PDF documents are processed and divided into smaller text chunks.

Rank 3
Distance: 1.229543
Content: The chunking system uses a chunk size of 500 characters with an overlap of 50 characters.


In [50]:
def improved_retrieve(query, top_k=3, threshold=1.2):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = improved_index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        if idx == -1:
            continue
        if distance <= threshold:
            retrieved_chunks.append({
                "rank": rank,
                "content": improved_chunks[idx],
                "similarity_score": float(distance)
           })
    return retrieved_chunks

In [51]:
def improved_diagnostic_rag(query, top_k=3, threshold=1.2):
    retrieved_chunks = improved_retrieve(
        query,
        top_k=top_k,
        threshold=threshold
    )
    if not retrieved_chunks:
        return {
            "query": query,
            "retrieved_chunks": [],
            "answer": "I don't have enough information in the provided context."
        }
    context = "\n\n".join(
        chunk["content"]
        for chunk in retrieved_chunks
    )
    answer = generate_answer(
        context,
        query
    )
    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer
    }

In [52]:
result = improved_diagnostic_rag(
    "What is the maximum file size allowed for PDF uploads?"
)
print("QUERY:")
print(result["query"])
print("\nRETRIEVED CHUNKS:")
for chunk in result["retrieved_chunks"]:
    print("\nRank:", chunk["rank"])
    print("Distance:", chunk["similarity_score"])
    print("Content:", chunk["content"])
print("\nFINAL ANSWER:")
print(result["answer"])

QUERY:
What is the maximum file size allowed for PDF uploads?

RETRIEVED CHUNKS:

Rank: 1
Distance: 0.3051392436027527
Content: Users can upload PDF files with a maximum file size of 10 MB.

Rank: 2
Distance: 0.8626115322113037
Content: Uploaded PDF documents are processed and divided into smaller text chunks.

FINAL ANSWER:
Users can upload PDF files with a maximum file size of 10 MB.


In [53]:
improved_results = []
for i, test in enumerate(test_queries, start=1):
    print(f"Running improved query {i}/15...")
    result = improved_diagnostic_rag(
        test["query"]
    )
    result["test_id"] = i
    result["target_failure"] = test["target_failure"]
    improved_results.append(result)
print("\nCompleted:", len(improved_results), "queries")

Running improved query 1/15...
Running improved query 2/15...
Running improved query 3/15...
Running improved query 4/15...
Running improved query 5/15...
Running improved query 6/15...
Running improved query 7/15...
Running improved query 8/15...
Running improved query 9/15...
Running improved query 10/15...
Running improved query 11/15...
Running improved query 12/15...
Running improved query 13/15...
Running improved query 14/15...
Running improved query 15/15...

Completed: 15 queries


In [54]:
improved_scorecard = []

for result in improved_results:

    test_id = result["test_id"]
    expected = expected_answers[test_id]

    r_score = retrieval_score(
        result,
        expected
    )

    a_score = answer_score(
        result,
        expected
    )

    improved_scorecard.append({
        "test_id": test_id,
        "query": result["query"],
        "retrieval_quality": r_score,
        "answer_quality": a_score,
        "answer": result["answer"]
    })

improved_scorecard_df = pd.DataFrame(
    improved_scorecard
)

improved_scorecard_df

,test_id,query,retrieval_quality,answer_quality,answer
0,1,What is the refund policy for cancelled orders?,5,5,The refund policy allows users to request a re...
1,2,Which database is used to store user authentic...,5,1,User authentication is implemented using JSON ...
2,3,What is the maximum file size allowed for uplo...,5,5,Users can upload PDF files with a maximum file...
3,4,Give me a complete summary of all the system f...,1,1,I don't have enough information in the provide...
4,5,Explain everything about the application's arc...,2,2,"The application provides document upload, sema..."
5,6,Describe all the technical requirements and im...,1,1,I don't have enough information in the provide...
6,7,"According to the retrieved documents, what pro...",5,5,The application uses Python as its primary pro...
7,8,"According to the documentation, what authentic...",5,1,The authentication API endpoint is /api/auth/l...
8,9,"According to the provided context, what is the...",5,5,The backend exposes REST APIs for communicatio...
9,10,Tell me about the system.,1,1,I don't have enough information in the provide...


In [55]:
improved_avg_retrieval = (
    improved_scorecard_df["retrieval_quality"].mean()
)

improved_avg_answer = (
    improved_scorecard_df["answer_quality"].mean()
)

print(
    f"Improved Retrieval Quality: "
    f"{improved_avg_retrieval:.2f}/5"
)

print(
    f"Improved Answer Quality: "
    f"{improved_avg_answer:.2f}/5"
)

Improved Retrieval Quality: 3.47/5
Improved Answer Quality: 2.93/5


In [56]:
comparison = pd.DataFrame({
    "Metric": [
        "Average Retrieval Quality",
        "Average Answer Quality"
    ],

    "Before Fix": [
        average_retrieval,
        average_answer
    ],

    "After Fix": [
        improved_avg_retrieval,
        improved_avg_answer
    ]
})

comparison

,Metric,Before Fix,After Fix
0,Average Retrieval Quality,3.733333,3.466667
1,Average Answer Quality,2.933333,2.933333


In [57]:
with open(
    "rag_improved_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        improved_results,
        f,
        indent=4,
        ensure_ascii=False
    )
print("Saved: rag_improved_results.json")

Saved: rag_improved_results.json


In [58]:
trace_queries = [
    "Which database is used to store user authentication information?",
    "What is the exact API endpoint used for authentication?"
]

for query in trace_queries:

    print("=" * 80)
    print("QUERY:", query)

    print("\n--- BASELINE RETRIEVAL ---")

    baseline = diagnostic_rag(query)

    for chunk in baseline["retrieved_chunks"]:
        print(
            f"\nRank {chunk['rank']} "
            f"| Distance: {chunk['similarity_score']:.4f}"
        )
        print(chunk["content"])

    print("\n--- IMPROVED RETRIEVAL ---")

    improved = improved_diagnostic_rag(query)

    for chunk in improved["retrieved_chunks"]:
        print(
            f"\nRank {chunk['rank']} "
            f"| Distance: {chunk['similarity_score']:.4f}"
        )
        print(chunk["content"])

QUERY: Which database is used to store user authentication information?

--- BASELINE RETRIEVAL ---

Rank 1 | Distance: 0.9136
User authentication is implemented using JSON Web Tokens (JWT).

Rank 2 | Distance: 1.0024
PostgreSQL is used as the primary relational database for storing application data.

Rank 3 | Distance: 1.1740
The authentication API endpoint is /api/auth/login.

--- IMPROVED RETRIEVAL ---

Rank 1 | Distance: 0.9136
User authentication is implemented using JSON Web Tokens (JWT).

Rank 2 | Distance: 1.0024
PostgreSQL is used as the primary relational database for storing application data.

Rank 3 | Distance: 1.1740
The authentication API endpoint is /api/auth/login.
QUERY: What is the exact API endpoint used for authentication?

--- BASELINE RETRIEVAL ---

Rank 1 | Distance: 0.3718
The authentication API endpoint is /api/auth/login.

Rank 2 | Distance: 0.5713
The registration API endpoint is /api/auth/register.

Rank 3 | Distance: 0.6777
User authentication is implemente

In [59]:
failure_traces = [
    {
        "trace_id": 1,
        "query": trace_queries[0],
        "cause": "Semantic overlap between authentication terminology and database/authentication chunks can cause FAISS to rank conceptually related chunks highly.",
        "decision": "The embedding model represents semantic meaning rather than exact keyword matching.",
        "fix": "Smaller overlapping chunks and similarity filtering reduce irrelevant context."
    },
    {
        "trace_id": 2,
        "query": trace_queries[1],
        "cause": "The word authentication is strongly associated with the JWT chunk, which can compete with the chunk containing the exact login endpoint.",
        "decision": "Embedding-based retrieval may prioritize topical similarity over exact factual attributes.",
        "fix": "Smaller chunks isolate the endpoint information and similarity filtering removes weaker matches."
    }
]

print("Failure traces created:", len(failure_traces))

Failure traces created: 2


In [60]:
import os
import json
from datetime import datetime
LOG_FILE = "rag_diagnostic_log.json"
def save_rag_log(result):
    if os.path.exists(LOG_FILE):
        with open(
            LOG_FILE,
            "r",
            encoding="utf-8"
        ) as f:
            logs = json.load(f)
    else:
        logs = []
    result["timestamp"] = datetime.now().isoformat()
    logs.append(result)
    with open(
        LOG_FILE,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            logs,
            f,
            indent=4,
            ensure_ascii=False
        )
    return result
print("Automatic logging ready!")

Automatic logging ready!


In [61]:
def diagnostic_rag(query, top_k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(
        query_embedding,
        top_k
    )
    retrieved_chunks = []
    for rank, (idx, distance) in enumerate(
        zip(indices[0], distances[0]),
        start=1
    ):
        if idx == -1:
            continue
        retrieved_chunks.append({
            "rank": rank,
            "content": documents[idx],
            "similarity_score": float(distance)
        })
    context = "\n\n".join(
        chunk["content"]
        for chunk in retrieved_chunks
    )
    answer = generate_answer(
        context,
        query
    )
    result = {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "answer": answer
    }
    save_rag_log(result)
    return result

In [62]:
test_result = diagnostic_rag(
    "What database is used by the application?"
)
print("Answer:")
print(test_result["answer"])
print("\nLog file created:", os.path.exists(LOG_FILE))

Answer:
PostgreSQL is used as the primary relational database for storing application data.

Log file created: True


In [63]:
with open(
    "rag_diagnostic_log.json",
    "r",
    encoding="utf-8"
) as f:
    logs = json.load(f)
print("Number of logged calls:", len(logs))
print("\nLatest logged query:")
print(logs[-1]["query"])
print("\nRetrieved chunks:")
for chunk in logs[-1]["retrieved_chunks"]:
    print(
        chunk["rank"],
        chunk["similarity_score"],
        chunk["content"]
    )

Number of logged calls: 1

Latest logged query:
What database is used by the application?

Retrieved chunks:
1 0.6241037845611572 PostgreSQL is used as the primary relational database for storing application data.
2 1.0454661846160889 The application provides document upload, semantic search, authentication, and course generation features.
3 1.1311616897583008 Redis is used for caching frequently accessed data and improving application performance.


In [64]:
final_analysis = []

for result in all_results:

    test_id = result["test_id"]
    expected = expected_answers[test_id]

    retrieval = retrieval_score(
        result,
        expected
    )

    answer = answer_score(
        result,
        expected
    )

    failure = classify_failure(
        result,
        expected
    )

    # Create diagnosis
    if failure == "retrieval_failure":

        diagnosis = (
            "The correct information was not found in the "
            "top retrieved chunks."
        )

    elif failure == "answer_context_mismatch":

        diagnosis = (
            "The relevant information was retrieved, but "
            "the generated answer did not use it correctly."
        )

    elif failure == "correct_retrieval":

        diagnosis = (
            "The relevant chunk was retrieved and the answer "
            "was supported by the retrieved context."
        )

    else:

        diagnosis = (
            "The result requires manual inspection."
        )

    final_analysis.append({
        "Test ID": test_id,
        "Query": result["query"],
        "Failure Type": failure,
        "Retrieval Quality": retrieval,
        "Answer Quality": answer,
        "Diagnosis": diagnosis
    })


final_analysis_df = pd.DataFrame(final_analysis)

final_analysis_df

,Test ID,Query,Failure Type,Retrieval Quality,Answer Quality,Diagnosis
0,1,What is the refund policy for cancelled orders?,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
1,2,Which database is used to store user authentic...,answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
2,3,What is the maximum file size allowed for uplo...,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
3,4,Give me a complete summary of all the system f...,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
4,5,Explain everything about the application's arc...,correct_retrieval,2,2,The relevant chunk was retrieved and the answe...
5,6,Describe all the technical requirements and im...,retrieval_failure,1,1,The correct information was not found in the t...
6,7,"According to the retrieved documents, what pro...",correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
7,8,"According to the documentation, what authentic...",answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
8,9,"According to the provided context, what is the...",answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
9,10,Tell me about the system.,retrieval_failure,1,1,The correct information was not found in the t...


In [65]:
print(
    "Average Retrieval Quality:",
    round(
        final_analysis_df["Retrieval Quality"].mean(),
        2
    ),
    "/5"
)

print(
    "Average Answer Quality:",
    round(
        final_analysis_df["Answer Quality"].mean(),
        2
    ),
    "/5"
)

Average Retrieval Quality: 3.73 /5
Average Answer Quality: 2.93 /5


In [66]:
failure_counts = (
    final_analysis_df["Failure Type"]
    .value_counts()
)

print(failure_counts)

Failure Type
correct_retrieval          8
retrieval_failure          4
answer_context_mismatch    3
Name: count, dtype: int64


In [67]:
print("Baseline results:", len(all_results))
print("Improved results:", len(improved_results))
print("\nBaseline average retrieval:",
      round(average_retrieval, 2), "/5")
print("Baseline average answer:",
      round(average_answer, 2), "/5")
print("\nImproved average retrieval:",
      round(improved_avg_retrieval, 2), "/5")
print("Improved average answer:",
      round(improved_avg_answer, 2), "/5")

Baseline results: 15
Improved results: 15

Baseline average retrieval: 3.73 /5
Baseline average answer: 2.93 /5

Improved average retrieval: 3.47 /5
Improved average answer: 2.93 /5


In [68]:
final_analysis_df

,Test ID,Query,Failure Type,Retrieval Quality,Answer Quality,Diagnosis
0,1,What is the refund policy for cancelled orders?,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
1,2,Which database is used to store user authentic...,answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
2,3,What is the maximum file size allowed for uplo...,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
3,4,Give me a complete summary of all the system f...,correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
4,5,Explain everything about the application's arc...,correct_retrieval,2,2,The relevant chunk was retrieved and the answe...
5,6,Describe all the technical requirements and im...,retrieval_failure,1,1,The correct information was not found in the t...
6,7,"According to the retrieved documents, what pro...",correct_retrieval,5,5,The relevant chunk was retrieved and the answe...
7,8,"According to the documentation, what authentic...",answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
8,9,"According to the provided context, what is the...",answer_context_mismatch,5,1,"The relevant information was retrieved, but th..."
9,10,Tell me about the system.,retrieval_failure,1,1,The correct information was not found in the t...


In [69]:
print(final_analysis_df.to_string(index=False))

 Test ID                                                                                                    Query            Failure Type  Retrieval Quality  Answer Quality                                                                                  Diagnosis
       1                                                          What is the refund policy for cancelled orders?       correct_retrieval                  5               5    The relevant chunk was retrieved and the answer was supported by the retrieved context.
       2                                         Which database is used to store user authentication information? answer_context_mismatch                  5               1 The relevant information was retrieved, but the generated answer did not use it correctly.
       3                                                       What is the maximum file size allowed for uploads?       correct_retrieval                  5               5    The relevant chunk was retrieved

In [70]:
final_analysis_df.to_csv(
    "rag_failure_scorecard.csv",
    index=False
)
print("Saved: rag_failure_scorecard.csv")

Saved: rag_failure_scorecard.csv


In [71]:
with open(
    "failure_traces.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        failure_traces,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved: failure_traces.json")

Saved: failure_traces.json


In [72]:
import os

files = [
    "rag_results.json",
    "rag_improved_results.json",
    "rag_diagnostic_log.json",
    "rag_failure_scorecard.csv",
    "failure_traces.json"
]

for file in files:
    print(
        file,
        "→",
        "FOUND" if os.path.exists(file) else "MISSING"
    )

rag_results.json → FOUND
rag_improved_results.json → FOUND
rag_diagnostic_log.json → FOUND
rag_failure_scorecard.csv → FOUND
failure_traces.json → FOUND
